### Get file size first

In [9]:
import os
for f in ['logon.csv', 'device.csv', 'file.csv', 'email.csv', 'http.csv']:
    path = f"/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/{f}"
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f"{f}: {size_mb:.1f} MB")

logon.csv: 55.8 MB
device.csv: 27.6 MB
file.csv: 184.1 MB
email.csv: 1299.0 MB
http.csv: 13862.9 MB


### For logon, device, file — load normally, no issue

In [4]:
import pandas as pd
logon = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/logon.csv")
device = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/device.csv")
file_activity = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/file.csv")

for df, name in [(logon,"logon"), (device,"device"), (file_activity,"file")]:
    df['date'] = pd.to_datetime(df['date'])
    print(f"{name}: {df.shape}")

logon: (854859, 5)
device: (405380, 5)
file: (445581, 6)


### For email.csv (1.3GB) — load only needed columns

In [6]:
# First peek at columns without loading everything
email_cols = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/email.csv", nrows=5)
print(email_cols.columns.tolist())

['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments', 'content']


In [10]:
email = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/email.csv", usecols=['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments'])
email['date'] = pd.to_datetime(email['date'])
print(email.shape)

(2629979, 10)


### For http.csv (13.9GB) — process in chunks, extract only what you need
Don't load this fully into memory. Instead, read it in chunks and immediately aggregate down to something small (e.g., per user-day counts), discarding the rest as you go:
This reads the file piece by piece (1 million rows at a time), immediately collapses each piece down to just "how many http events per user per day," and throws away the raw detail. End result: a small, manageable table instead of a 14GB dataframe in memory.

In [13]:
chunk_size = 1_000_000
http_daily_counts = []

for chunk in pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/http.csv", usecols=['date', 'user', 'url'], chunksize=chunk_size):
    chunk['date'] = pd.to_datetime(chunk['date'])
    chunk['day'] = chunk['date'].dt.date
    daily = chunk.groupby(['user', 'day']).size().reset_index(name='http_count')
    http_daily_counts.append(daily)
    print("Processed a chunk...")

http_daily = pd.concat(http_daily_counts).groupby(['user','day'])['http_count'].sum().reset_index()
print(http_daily.shape)
print(http_daily.head())

Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
Processed a chunk...
(329845, 3)
      user         day  http_count
0  AAE0190  2010-01-04         143
1  AAE0190  2010-01-05         143
2  AAE0190  2010-01-06         143
3  AAE0190  2010-01-07         143
4  AAE0190  2010-01-08         143


In [14]:
http_daily.to_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/http_daily_counts.csv", index=False)

### check one user's raw data directly (bypass chunking, just filter):

In [18]:
sample_check = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/http.csv", usecols=['date', 'user', 'url'])
sample_check['date'] = pd.to_datetime(sample_check['date'])
sample_check['day'] = sample_check['date'].dt.date

user_check = sample_check[sample_check['user'] == 'AAE0190']
print(user_check.groupby('day').size())

day
2010-01-04    143
2010-01-05    143
2010-01-06    143
2010-01-07    143
2010-01-08    143
             ... 
2011-05-10    143
2011-05-11    143
2011-05-12    143
2011-05-13    143
2011-05-16    143
Length: 346, dtype: int64


### Data Quality Observation: HTTP Activity Patterns
Several users (e.g., AAE0190) show identical daily HTTP event counts 
(exactly 143/day) across the entire ~17-month period, suggesting the 
synthetic data generator assigned some baseline users a fixed, repeating 
browsing template rather than naturally varying behavior. This is a known 
characteristic of simulated datasets and is noted as a limitation: models 
trained on this data may learn "rigid repetition" as the definition of 
normal, which could affect generalization to real-world log data where 
normal behavior naturally varies day to day.

### Aggregate every source down to user_day level using the same chunking strategy where needed
For the small files (logon, device, file — all under 200MB), just load normally and aggregate directly:

In [20]:
logon['day'] = logon['date'].dt.date
logon_daily = logon.groupby(['user','day']).size().reset_index(name='logon_count')

device['day'] = device['date'].dt.date
device_daily = device.groupby(['user','day']).size().reset_index(name='device_count')

file_activity['day'] = file_activity['date'].dt.date
file_daily = file_activity.groupby(['user','day']).size().reset_index(name='file_count')

For email.csv (1.3GB — medium, can likely load directly with usecols, or chunk if needed):

In [21]:
email['day'] = email['date'].dt.date
email_daily = email.groupby(['user','day']).size().reset_index(name='email_count')

For http.csv (already built above as http_daily) just rename for consistency:

In [22]:
http_daily = http_daily.rename(columns={'day':'day'}) 

### Merge all daily aggregates into one master table

In [23]:
master = logon_daily.merge(device_daily, on=['user','day'], how='outer')
master = master.merge(file_daily, on=['user','day'], how='outer')
master = master.merge(email_daily, on=['user','day'], how='outer')
master = master.merge(http_daily, on=['user','day'], how='outer')
master = master.fillna(0)

print(master.shape)
print(master.head())

(330452, 7)
      user         day  logon_count  device_count  file_count  email_count  \
0  AAE0190  2010-01-04            2           0.0         0.0         14.0   
1  AAE0190  2010-01-05            2           0.0         0.0         13.0   
2  AAE0190  2010-01-06            2           0.0         0.0         14.0   
3  AAE0190  2010-01-07            2           0.0         0.0         14.0   
4  AAE0190  2010-01-08            2           0.0         0.0         13.0   

   http_count  
0       143.0  
1       143.0  
2       143.0  
3       143.0  
4       143.0  


In [24]:
master.to_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/user_day_master.csv", index=False)